# TIPy Double Inverted Pendulum — V2 Experimental Pipeline

## SECTION 0 — Experiment overview

Goal:

1. Run a classical nonlinear swing-up + linear stabilizing controller.
2. Verify the expert independently.
3. Record an expert video.
4. Generate expert demonstrations using MJX.
5. Behavior-clone the expert.
6. Evaluate and record the BC policy.
7. Initialize PPO from BC weights.
8. Fine-tune using the existing GPU PPO implementation.
9. Compare Expert vs BC vs BC→PPO.

```text
MuJoCo plant
     ↓
Nonlinear expert
energy shaping / feedback linearization
     ↓
linear MPC or LQR catch
     ↓
expert evaluation
     ↓
expert video
     ↓
MJX batched demonstrations
     ↓
Behavior Cloning
     ↓
BC evaluation + video
     ↓
PPO initialization from BC
     ↓
MJX PPO fine-tuning
     ↓
final comparison
```

Experimental question: **Can the classical controller solve the system? Can a neural policy imitate it? Can PPO improve beyond the imitation-initialized policy?**


## SECTION 1 — Setup

Imports are grouped as standard Python, NumPy/SciPy, MuJoCo, JAX/MJX, Flax/Optax, plotting, and rendering. Controller definitions appear later.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Recommended by the current MJX docs for NVIDIA GPU performance.
os.environ["XLA_FLAGS"] = (
    os.environ.get("XLA_FLAGS", "")
    + " --xla_gpu_triton_gemm_any=true"
)

if IN_COLAB or IN_KAGGLE:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--upgrade", "-q",
            "jax[cuda12]",
            "mujoco-mjx",
            "flax==0.12.8",
            "optax==0.2.8",
            "scipy>=1.11",
            "pillow>=10",
            "pandas>=2.0",
            "matplotlib>=3.8",
            "imageio>=2.34",
            "imageio-ffmpeg>=0.5",
        ],
        check=True,
    )

    subprocess.run(
            [
                "apt-get", "install", "-y", "-qq",
                "libegl1",
                "libglvnd0",
                "libglx0",
                "libegl-dev",
                "libgles2",
            ],
            check=True,
        )

# These MUST be set before mujoco, dm_control, brax, etc. are imported.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# JAX
os.environ["XLA_FLAGS"] = (
    os.environ.get("XLA_FLAGS", "")
    + " --xla_gpu_triton_gemm_any=true"
)

print("MUJOCO_GL =", os.environ["MUJOCO_GL"])
print("PYOPENGL_PLATFORM =", os.environ["PYOPENGL_PLATFORM"])

import mujoco

print("MuJoCo version:", mujoco.__version__)

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
import flax
import flax.linen as nn
from flax.training import train_state
import optax

print("JAX:", jax.__version__)
print("MuJoCo:", mujoco.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())

gpu_devices = [d for d in jax.devices() if d.platform == "gpu"]
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime → Change runtime type → GPU, "
        "restart the runtime, then run from the top."
    )


# Standard Python
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import csv
import json
import pickle
import time

# NumPy / SciPy
from scipy.linalg import solve_discrete_are

# Plotting and rendering
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
from PIL import Image, ImageDraw
from IPython.display import Video, display


## SECTION 2 — Configuration

The five blocks below are the experiment's single configuration surface. PPO defaults are unchanged from the working notebook. Actions are normalized to `[-1, 1]`; the environment converts them to force in newtons.


In [ ]:
SYSTEM_CONFIG = {
    "seed": 42,
    "max_episode_steps": 5000,
    "action_limit_n": 75.0,
    "rail_limit_m": 2.0,
}
EXPERT_CONFIG = {
    "energy_gain": 12.0,
    "cart_kp": 5.0,
    "cart_kd": 3.0,
    "virtual_force_limit": 75.0,
    "capture_angle_rad": np.deg2rad(30.0),
    "capture_rate_rad_s": 4.0,
    "lqr_q": (8.0, 180.0, 120.0, 3.0, 18.0, 12.0),
    "lqr_r": 0.08,
    "evaluation_episodes": 32,
}
BC_CONFIG = {
    "updates": 80,
    "epochs": 4,
    "num_minibatches": 16,
    "learning_rate": 3e-4,
    "validation_fraction": 0.1,
}
PPO_CONFIG = {
    "num_envs": 2048,
    "rollout_steps": 64,
    "total_steps": 300_000_000,
    "learning_rate": 3e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_eps": 0.2,
    "value_coef": 0.5,
    "entropy_coef": 0.0001,
    "max_grad_norm": 0.5,
    "epochs": 6,
    "num_minibatches": 16,
    "resume": True,
}
RENDER_CONFIG = {
    "width": 640,
    "height": 480,
    "fps": 50,
    "seconds": 10.0,
    "camera": "replay",
}

OUTPUT_DIR = Path(os.environ.get("TIPY_OUTPUT_DIR", Path.cwd() / "outputs"))
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
VIDEO_DIR = OUTPUT_DIR / "videos"
PLOT_DIR = OUTPUT_DIR / "plots"
METRIC_DIR = OUTPUT_DIR / "metrics"
for directory in (CHECKPOINT_DIR, VIDEO_DIR, PLOT_DIR, METRIC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BC_CHECKPOINT_PATH = CHECKPOINT_DIR / "bc_pretrained.pkl"
PPO_LATEST_PATH = CHECKPOINT_DIR / "ppo_latest.pkl"
PPO_BEST_PATH = CHECKPOINT_DIR / "ppo_best.pkl"
METRICS_PATH = METRIC_DIR / "ppo_metrics.csv"
DASHBOARD_PATH = PLOT_DIR / "training_dashboard.png"
REPLAY_PATH = VIDEO_DIR / "ppo_finetuned.mp4"

SEED = SYSTEM_CONFIG["seed"]
MAX_EPISODE_STEPS = SYSTEM_CONFIG["max_episode_steps"]
ACTION_LIMIT = SYSTEM_CONFIG["action_limit_n"]
RAIL_LIMIT = SYSTEM_CONFIG["rail_limit_m"]
NUM_ENVS = PPO_CONFIG["num_envs"]
ROLLOUT_STEPS = PPO_CONFIG["rollout_steps"]
TOTAL_STEPS = PPO_CONFIG["total_steps"]
LEARNING_RATE = PPO_CONFIG["learning_rate"]
GAMMA = PPO_CONFIG["gamma"]
GAE_LAMBDA = PPO_CONFIG["gae_lambda"]
CLIP_EPS = PPO_CONFIG["clip_eps"]
VALUE_COEF = PPO_CONFIG["value_coef"]
ENTROPY_COEF = PPO_CONFIG["entropy_coef"]
MAX_GRAD_NORM = PPO_CONFIG["max_grad_norm"]
PPO_EPOCHS = PPO_CONFIG["epochs"]
NUM_MINIBATCHES = PPO_CONFIG["num_minibatches"]
BC_UPDATES = BC_CONFIG["updates"]
BC_EPOCHS = BC_CONFIG["epochs"]
BC_NUM_MINIBATCHES = BC_CONFIG["num_minibatches"]
BC_LEARNING_RATE = BC_CONFIG["learning_rate"]
LOG_EVERY_UPDATES = 1
CHECKPOINT_EVERY_UPDATES = 10
RUN_CPU_EXPERT_VALIDATION = True
RUN_BC_TRAINING = True
RUN_PPO_TRAINING = True

BATCH_SIZE = NUM_ENVS * ROLLOUT_STEPS
MINIBATCH_SIZE = BATCH_SIZE // NUM_MINIBATCHES
BC_BATCH_SIZE = BATCH_SIZE
BC_MINIBATCH_SIZE = BC_BATCH_SIZE // BC_NUM_MINIBATCHES
assert BATCH_SIZE % NUM_MINIBATCHES == 0
assert BC_BATCH_SIZE % BC_NUM_MINIBATCHES == 0
print("output:", OUTPUT_DIR)
print("transitions/update:", BATCH_SIZE)


## SECTION 3 — Classical-controller source and Python conversion

Offline reference used during conversion: `johnglennII/Double-Inverted-Pendulum-Cart`, commit `6e25d9aa5da3fa2c3504bc0da4f9d1de51395c5f`.

The notebook contains self-contained Python/JAX implementations of the symbolic feedback-linearization equations, total mechanical energy, passivity-based virtual control, `u = alpha + beta*v`, force saturation, and hybrid switching logic. The local catch controller is recomputed from the MuJoCo plant with discrete LQR.

The notebook performs no repository cloning, external controller execution, file discovery, or runtime source loading. Colab needs only this notebook and its Python packages.


## SECTION 4 — MuJoCo model

The MuJoCo dynamics match the source controller model: `mc=0.4 kg`, `m1=m2=0.15 kg`, `L1=L2=0.5 m`, COM distances `0.25 m`, rod inertias `0.003125 kg·m²`, zero damping, gravity `9.81 m/s²`, and a `±75 N` cart-force range matching the local-controller bounds. The cart slide has no MuJoCo joint limit. The two yellow rail markers are visual boundaries; the environment handles `|x| >= 2 m` termination. The tall decorative side columns were removed. `mj_model` is the CPU model and `mjx_model` is the JAX model.


In [ ]:
MODEL_XML = r"""
<mujoco model="cartpole_double">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option
      timestep="0.005"
      gravity="0 0 -9.81"
      integrator="RK4"
      solver="Newton"
      iterations="1"
      ls_iterations="2"
      jacobian="dense">
    <flag eulerdamp="disable"/>
  </option>

  <default>
    <geom contype="0" conaffinity="0" />
  </default>

  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>

  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole1_mat" rgba="0.2 0.75 0.3 1" />
    <material name="pole2_mat" rgba="0.2 0.3 0.75 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>

  <worldbody>
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />

    <body name="frame">
      <geom name="rail" type="box" pos="0 0 0.85" size="2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>

    <body name="cart" pos="0 0 1">
      <joint
          name="cart_slide"
          type="slide"
          axis="1 0 0"
          limited="false"
          frictionloss="0"
          damping="0" />
      <inertial pos="0 0 0" mass="0.4" diaginertia="0.001 0.001 0.001" />
      <geom name="cart_geom" type="box" size="0.085 0.08 0.1" mass="0.4" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0"
            axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />

      <body name="pole1" pos="0 0.11 0">
        <joint name="pole1_hinge" type="hinge" axis="0 -1 0"
               frictionloss="0" damping="0" limited="false" />
        <inertial pos="0 0 -0.25" mass="0.15"
                  diaginertia="0.003125 0.003125 0.00001" />
        <site name="pole1_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole1_geom" type="box" pos="0 0 -0.25"
              size="0.02 0.01 0.25" mass="0.15" material="pole1_mat" />
        <site name="pole1_tip_site" pos="0 0 -0.5" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole2_mount_pin" type="capsule" pos="0 0.015 -0.5"
              axisangle="1 0 0 1.5708" size="0.012 0.015" material="metal_mat" />

        <body name="pole2" pos="0 0.03 -0.5">
          <joint name="pole2_hinge" type="hinge" axis="0 -1 0"
                 frictionloss="0" damping="0" ref="0" limited="false" />
          <inertial pos="0 0 -0.25" mass="0.15"
                    diaginertia="0.003125 0.003125 0.00001" />
          <site name="pole2_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
          <geom name="pole2_geom" type="box" pos="0 0 -0.25"
                size="0.02 0.01 0.25" mass="0.15" material="pole2_mat" />
          <site name="pole2_tip_site" pos="0 0 -0.5" size="0.015" type="sphere" material="site_mat" />
        </body>
      </body>
    </body>

    <camera name="replay" pos="0 6 1.4" fovy="50"
            xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>

  <actuator>
    <motor name="cart_motor" joint="cart_slide"
           gear="1" ctrlrange="-75 75" forcerange="-75 75" />
  </actuator>
</mujoco>
"""

mj_model = mujoco.MjModel.from_xml_string(MODEL_XML)

# Pure JAX MJX implementation. This is intentionally not the old CPU MjData.
mjx_model = mjx.put_model(mj_model, impl="jax")

print("nq:", mj_model.nq, "nv:", mj_model.nv, "nu:", mj_model.nu)
print("timestep:", mj_model.opt.timestep)


## SECTION 5 — State convention

Canonical physical controller state, in SI units:

```text
[x, x_dot, theta1, theta1_dot, theta2, theta2_dot]
```

MuJoCo uses `qpos = [x, q1, q2_relative]` and `qvel = [x_dot, q1_dot, q2_relative_dot]`. In this MuJoCo plant, an absolute angle of `π` is upright. The converted classical controller instead defines upright as zero. Both controller angles below are therefore absolute **upright-error angles**, positive toward negative world-x:

```text
theta1     = wrap(qpos[1] - π)
theta2     = wrap(qpos[1] + qpos[2] - π)
theta1_dot = qvel[1]
theta2_dot = qvel[1] + qvel[2]
```

Thus MuJoCo `(π, 0 relative)` maps to controller `(0, 0)` upright, while MuJoCo `(0, 0 relative)` maps to controller `(-π, -π)` down-down. The offset and its inverse occur only in `physical_state` and `physical_to_mujoco_state`; all observations, rewards, expert equations, LQR, evaluation, and rendering consume the canonical controller convention.


In [ ]:
from flax import struct

OBS_DIM = 8
ACTION_DIM = 1
MUJOCO_UPRIGHT_ANGLE = float(np.pi)

@struct.dataclass
class EnvState:
    data: object
    step_count: jax.Array
    episode_return: jax.Array
    episode_length: jax.Array
    captured_upright: jax.Array

def wrap_angle(x):
    return (x + jnp.pi) % (2.0 * jnp.pi) - jnp.pi

def physical_state(data):
    x = data.qpos[..., 0]
    relative1 = data.qpos[..., 1]
    relative2 = data.qpos[..., 2]

    dx = data.qvel[..., 0]
    relative_speed1 = data.qvel[..., 1]
    relative_speed2 = data.qvel[..., 2]

    theta1 = wrap_angle(relative1 - MUJOCO_UPRIGHT_ANGLE)
    theta2 = wrap_angle(relative1 + relative2 - MUJOCO_UPRIGHT_ANGLE)

    dtheta1 = relative_speed1
    dtheta2 = relative_speed1 + relative_speed2

    return x, dx, theta1, dtheta1, theta2, dtheta2



def physical_state_vector(data):
    return jnp.stack(physical_state(data), axis=-1)

def physical_to_mujoco_state(state, xp=np):
    x, x_dot, theta1, theta1_dot, theta2, theta2_dot = [state[..., i] for i in range(6)]
    return xp.stack(
        [x, theta1 + MUJOCO_UPRIGHT_ANGLE, theta2 - theta1, x_dot, theta1_dot, theta2_dot - theta1_dot],
        axis=-1,
    )

# Convention invariants: controller upright 0 <-> MuJoCo upright pi.
_upright_mj = physical_to_mujoco_state(np.zeros(6), np)
np.testing.assert_allclose(_upright_mj[:3], [0.0, np.pi, 0.0], atol=1e-7)


## SECTION 6 — MJX environment

The existing explicit MJX environment is preserved: batched reset, observation, reward, rail termination, time-limit truncation, and vmapped MJX stepping. PPO is not introduced here.


In [ ]:
def observation_from_data(data):
    x, dx, theta1, dtheta1, theta2, dtheta2 = physical_state(data)

    return jnp.stack(
        [
            jnp.clip(x, -RAIL_LIMIT, RAIL_LIMIT),
            jnp.clip(dx / 5.0, -1.0, 1.0),
            jnp.cos(theta1),
            jnp.sin(theta1),
            jnp.cos(theta2),
            jnp.sin(theta2),
            jnp.clip(dtheta1 / 20.0, -1.0, 1.0),
            jnp.clip(dtheta2 / 20.0, -1.0, 1.0),
        ],
        axis=-1,
    ).astype(jnp.float32)

def _reset_one(key):
    key_pos, key_ang1, key_ang2, key_vel, key_w1, key_w2 = jax.random.split(key, 6)

    data = mjx.make_data(mjx_model)

    # MuJoCo down-down is near absolute angle 0; physical_state maps it near -pi.
    absolute1 = jax.random.uniform(
        key_ang1, (), minval=-0.04, maxval=0.04
    )
    absolute2 = jax.random.uniform(
        key_ang2, (), minval=-0.04, maxval=0.04
    )

    qpos = jnp.array(
        [
            jax.random.uniform(key_pos, (), minval=-0.01, maxval=0.01),
            absolute1,
            absolute2 - absolute1,
        ],
        dtype=jnp.float32,
    )

    absolute_speed1 = jax.random.uniform(
        key_w1, (), minval=-0.02, maxval=0.02
    )
    absolute_speed2 = jax.random.uniform(
        key_w2, (), minval=-0.02, maxval=0.02
    )

    qvel = jnp.array(
        [
            jax.random.uniform(key_vel, (), minval=-0.01, maxval=0.01),
            absolute_speed1,
            absolute_speed2 - absolute_speed1,
        ],
        dtype=jnp.float32,
    )

    data = data.replace(qpos=qpos, qvel=qvel)
    data = mjx.forward(mjx_model, data)
    return data

reset_data_batch = jax.jit(jax.vmap(_reset_one))

def reset_batch(keys):
    data = reset_data_batch(keys)
    n = keys.shape[0]
    state = EnvState(
        data=data,
        step_count=jnp.zeros((n,), dtype=jnp.int32),
        episode_return=jnp.zeros((n,), dtype=jnp.float32),
        episode_length=jnp.zeros((n,), dtype=jnp.int32),
        captured_upright=jnp.zeros((n,), dtype=jnp.bool_),
    )
    return state, observation_from_data(data)

def _step_one(data, action):
    # action is already tanh-squashed to [-1, 1].
    force = action[0] * ACTION_LIMIT
    ctrl = data.ctrl.at[0].set(force)
    data = data.replace(ctrl=ctrl)
    data = mjx.step(mjx_model, data)
    return data

step_data_batch = jax.vmap(_step_one)

def step_batch(state, action):
    next_data = step_data_batch(state.data, action)

    x, dx, theta1, dtheta1, theta2, dtheta2 = physical_state(next_data)
    next_obs_terminal = observation_from_data(next_data)

    normalized_force = action[..., 0]

    angle_reward = 0.5 * (jnp.cos(theta1) + jnp.cos(theta2))
    position_penalty = 0.25 * (x / RAIL_LIMIT) ** 2
    velocity_penalty = 0.01 * dx**2
    angular_velocity_penalty = 0.003 * (dtheta1**2 + dtheta2**2)
    control_penalty = 0.001 * normalized_force**2

    upright = (jnp.abs(theta1) < 0.35) & (jnp.abs(theta2) < 0.35)
    upright_bonus = jnp.where(upright, 3.0, 0.0)

    reward = (
        angle_reward
        - position_penalty
        - velocity_penalty
        - angular_velocity_penalty
        - control_penalty
        + upright_bonus
    )

    next_step_count = state.step_count + 1
    terminated = jnp.abs(x) >= RAIL_LIMIT
    truncated = next_step_count >= MAX_EPISODE_STEPS
    done = terminated | truncated

    reward = reward - jnp.where(terminated, 100.0, 0.0)

    episode_return = state.episode_return + reward
    episode_length = state.episode_length + 1
    captured = state.captured_upright | upright

    return (
        next_data,
        next_obs_terminal,
        reward.astype(jnp.float32),
        terminated,
        truncated,
        done,
        episode_return,
        episode_length,
        captured,
    )


# Minimal environment sanity check.
_sanity_keys = jax.random.split(jax.random.PRNGKey(SEED), 4)
_sanity_state, _sanity_obs = reset_batch(_sanity_keys)
_sanity_result = step_batch(_sanity_state, jnp.zeros((4, ACTION_DIM), dtype=jnp.float32))
_sanity_physical = physical_state_vector(_sanity_state.data)
assert _sanity_obs.shape == (4, OBS_DIM)
assert _sanity_result[2].shape == (4,)
assert bool(jnp.all(jnp.cos(_sanity_physical[:, 2]) < -0.99))
assert bool(jnp.all(jnp.cos(_sanity_physical[:, 4]) < -0.99))
print("MJX sanity:", _sanity_obs.shape, _sanity_result[2].shape)


# PART II — CLASSICAL EXPERT

## SECTION 7 — Expert physical parameters

Values match the XML and converted controller model: kg, m, kg·m², N·s/m, N·m·s/rad, and m/s². Explicit MuJoCo inertials define the expert plant.


In [ ]:
@dataclass(frozen=True)
class ExpertParams:
    mc: float
    m1: float
    m2: float
    L1: float
    L2: float
    l1: float
    l2: float
    J1: float
    J2: float
    cc: float
    c1: float
    c2: float
    g: float

EXPERT_PARAMS = ExpertParams(
    mc=0.4,
    m1=0.15,
    m2=0.15,
    L1=0.50,
    L2=0.50,
    l1=0.25,
    l2=0.25,
    J1=0.003125,
    J2=0.003125,
    cc=0.0,
    c1=0.0,
    c2=0.0,
    g=9.81,
)

cart_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "cart")
pole1_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "pole1")
pole2_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "pole2")
np.testing.assert_allclose(mj_model.body_mass[[cart_id, pole1_id, pole2_id]], [0.4, 0.15, 0.15])
np.testing.assert_allclose(mj_model.body_inertia[[pole1_id, pole2_id], 1], [EXPERT_PARAMS.J1, EXPERT_PARAMS.J2])
np.testing.assert_allclose(mj_model.body_ipos[[pole1_id, pole2_id], 2], [-0.25, -0.25])
np.testing.assert_allclose(mj_model.body_pos[pole2_id, 2], -0.50)
np.testing.assert_allclose(mj_model.dof_damping, [0.0, 0.0, 0.0])
np.testing.assert_allclose(mj_model.actuator_ctrlrange[0], [-75.0, 75.0])
assert not bool(mj_model.jnt_limited[0]), "cart slide must remain physically unlimited"
pole1_tip_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SITE, "pole1_tip_site")
pole2_tip_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SITE, "pole2_tip_site")
_down_check = mujoco.MjData(mj_model)
_down_check.qpos[:] = [0.0, 0.0, 0.0]
mujoco.mj_forward(mj_model, _down_check)
assert _down_check.site_xpos[pole2_tip_id, 2] < _down_check.site_xpos[pole1_tip_id, 2] < 1.0
_up_check = mujoco.MjData(mj_model)
_up_check.qpos[:] = [0.0, np.pi, 0.0]
mujoco.mj_forward(mj_model, _up_check)
assert _up_check.site_xpos[pole2_tip_id, 2] > _up_check.site_xpos[pole1_tip_id, 2] > 1.0
print(EXPERT_PARAMS)


## SECTION 8 — Port `compute_alpha_beta`

The generated symbolic equations are preserved algebraically in Python. The derivation and control-law caller establish this convention:

```text
x_ddot = a(x) + b(x) u
alpha = -a/b, beta = 1/b
u = alpha(x) + beta(x) v  ⇒  x_ddot = v
```

`compute_alpha_beta` accepts NumPy or JAX through `xp`; there is no external source loading or SciPy call in the JIT path.


In [ ]:
def compute_alpha_beta(state, params, xp=np):
    """Python/JAX feedback-linearization map in canonical physical-state order."""
    mc, m1, m2 = params.mc, params.m1, params.m2
    L1, L2, l1, l2 = params.L1, params.L2, params.l1, params.l2
    J1, J2 = params.J1, params.J2
    cc, c1, c2, g = params.cc, params.c1, params.c2, params.g
    x1, x2, x3, x4, x5, x6 = [state[..., i] for i in range(6)]
    et1 = J1*J2*cc*x2*2.0+J2*l1**3*m1**2*x4**2*xp.sin(x3)*2.0+J1*l2**3*m2**2*x6**2*xp.sin(x5)*2.0+L1**2*cc*l2**2*m2**2*x2+J2*L1**2*cc*m2*x2*2.0+J2*cc*l1**2*m1*x2*2.0+J1*cc*l2**2*m2*x2*2.0-J2*L1**2*g*m2**2*xp.sin(x3*2.0)+J2*L1**3*m2**2*x4**2*xp.sin(x3)*2.0-J2*g*l1**2*m1**2*xp.sin(x3*2.0)-J1*g*l2**2*m2**2*xp.sin(x5*2.0)+J2*L1*c1*m2*x4*xp.cos(x3)*2.0+J2*L1*c2*m2*x4*xp.cos(x3)*2.0-J2*L1*c2*m2*x6*xp.cos(x3)*2.0-L1*c1*l2**2*m2**2*x4*xp.cos(x3-x5*2.0)-L1*c2*l2**2*m2**2*x4*xp.cos(x3-x5*2.0)
    et2 = L1*c2*l2**2*m2**2*x6*xp.cos(x3-x5*2.0)+J2*c1*l1*m1*x4*xp.cos(x3)*2.0+J2*c2*l1*m1*x4*xp.cos(x3)*2.0-J2*c2*l1*m1*x6*xp.cos(x3)*2.0-J1*c2*l2*m2*x4*xp.cos(x5)*2.0+J1*c2*l2*m2*x6*xp.cos(x5)*2.0+J1*L1*l2**2*m2**2*x4**2*xp.sin(x3)+J2*L1**2*l2*m2**2*x6**2*xp.sin(x5)+J1*J2*L1*m2*x4**2*xp.sin(x3)*2.0-J1*L1*l2**2*m2**2*x4**2*xp.sin(x3-x5*2.0)+J1*J2*l1*m1*x4**2*xp.sin(x3)*2.0+J1*J2*l2*m2*x6**2*xp.sin(x5)*2.0+L1**2*c2*l2*m2**2*x4*xp.cos(x3*2.0-x5)-L1**2*c2*l2*m2**2*x6*xp.cos(x3*2.0-x5)+cc*l1**2*l2**2*m1*m2*x2*2.0
    et3 = -g*l1**2*l2**2*m1**2*m2*xp.sin(x3*2.0)-g*l1**2*l2**2*m1*m2**2*xp.sin(x5*2.0)+J2*L1**2*l2*m2**2*x6**2*xp.sin(x3*2.0-x5)-L1**2*cc*l2**2*m2**2*x2*xp.cos(x3*2.0-x5*2.0)+l1**3*l2**2*m1**2*m2*x4**2*xp.sin(x3)*2.0+l1**2*l2**3*m1*m2**2*x6**2*xp.sin(x5)*2.0+L1*c1*l2**2*m2**2*x4*xp.cos(x3)+L1*c2*l2**2*m2**2*x4*xp.cos(x3)-L1*c2*l2**2*m2**2*x6*xp.cos(x3)-L1**2*c2*l2*m2**2*x4*xp.cos(x5)+L1**2*c2*l2*m2**2*x6*xp.cos(x5)+c1*l1*l2**2*m1*m2*x4*xp.cos(x3)*2.0
    et4 = c2*l1*l2**2*m1*m2*x4*xp.cos(x3)*2.0-c2*l1*l2**2*m1*m2*x6*xp.cos(x3)*2.0-c2*l1**2*l2*m1*m2*x4*xp.cos(x5)*2.0+c2*l1**2*l2*m1*m2*x6*xp.cos(x5)*2.0+L1*l1**2*l2**2*m1*m2**2*x4**2*xp.sin(x3)+L1**2*l1*l2**2*m1*m2**2*x4**2*xp.sin(x3)+J2*L1*l1**2*m1*m2*x4**2*xp.sin(x3)*2.0+J2*L1**2*l1*m1*m2*x4**2*xp.sin(x3)*2.0+L1*l1*l2**3*m1*m2**2*x6**2*xp.sin(x3*2.0-x5)-L1*l1**2*l2**2*m1*m2**2*x4**2*xp.sin(x3-x5*2.0)+L1**2*l1*l2**2*m1*m2**2*x4**2*xp.sin(x3-x5*2.0)+J1*l1*l2**2*m1*m2*x4**2*xp.sin(x3)*2.0
    et5 = J2*l1**2*l2*m1*m2*x6**2*xp.sin(x5)*2.0-L1*g*l1*l2**2*m1*m2**2*xp.sin(x3*2.0)+L1*g*l1*l2**2*m1*m2**2*xp.sin(x5*2.0)-L1*l1*l2**3*m1*m2**2*x6**2*xp.sin(x5)-J2*L1*g*l1*m1*m2*xp.sin(x3*2.0)*2.0+J2*L1*l1*l2*m1*m2*x6**2*xp.sin(x3*2.0-x5)+L1*c2*l1*l2*m1*m2*x4*xp.cos(x5)-L1*c2*l1*l2*m1*m2*x6*xp.cos(x5)-J2*L1*l1*l2*m1*m2*x6**2*xp.sin(x5)+L1*c2*l1*l2*m1*m2*x4*xp.cos(x3*2.0-x5)-L1*c2*l1*l2*m1*m2*x6*xp.cos(x3*2.0-x5)
    et6 = J2*L1**2*m2**2+J2*l1**2*m1**2+J1*l2**2*m2**2+J1*J2*m1*2.0+J1*J2*m2*2.0+J1*J2*mc*2.0+L1**2*l2**2*m1*m2**2+L1**2*l2**2*m2**2*mc+J2*L1**2*m1*m2*2.0+J2*L1**2*m2*mc*2.0+l1**2*l2**2*m1*m2**2+l1**2*l2**2*m1**2*m2+J1*l2**2*m1*m2*2.0+J2*l1**2*m1*m2*2.0+J2*l1**2*m1*mc*2.0+J1*l2**2*m2*mc*2.0-J2*L1**2*m2**2*xp.cos(x3*2.0)-J2*l1**2*m1**2*xp.cos(x3*2.0)-J1*l2**2*m2**2*xp.cos(x5*2.0)-L1*l1*l2**2*m1*m2**2-J2*L1*l1*m1*m2*2.0
    et7 = l1**2*l2**2*m1*m2*mc*2.0-l1**2*l2**2*m1**2*m2*xp.cos(x3*2.0)-l1**2*l2**2*m1*m2**2*xp.cos(x5*2.0)-L1**2*l2**2*m1*m2**2*xp.cos(x3*2.0-x5*2.0)-L1**2*l2**2*m2**2*mc*xp.cos(x3*2.0-x5*2.0)-L1*l1*l2**2*m1*m2**2*xp.cos(x3*2.0)+L1*l1*l2**2*m1*m2**2*xp.cos(x5*2.0)-J2*L1*l1*m1*m2*xp.cos(x3*2.0)*2.0+L1*l1*l2**2*m1*m2**2*xp.cos(x3*2.0-x5*2.0)
    denominator = J1*J2*2.0+L1**2*l2**2*m2**2+J2*L1**2*m2*2.0+J2*l1**2*m1*2.0+J1*l2**2*m2*2.0-L1**2*l2**2*m2**2*xp.cos(x3*2.0-x5*2.0)+l1**2*l2**2*m1*m2*2.0
    alpha = (et1 + et2 + et3 + et4 + et5) / denominator
    beta = (et6 + et7) / denominator
    return alpha, beta


## SECTION 9 — Energy computation

This is the source controller's exact total mechanical energy in canonical controller angles. Swing-up evaluates it after setting cart speed to zero. Controller upright is `(theta1, theta2) = (0, 0)`, corresponding to MuJoCo absolute angles `(π, π)`; controller down-down is `(-π, -π)`, corresponding to MuJoCo absolute angles `(0, 0)`.


In [ ]:
def compute_energy(state, params, xp=np):
    mc, m1, m2 = params.mc, params.m1, params.m2
    L1, L2, l1, l2 = params.L1, params.L2, params.l1, params.l2
    J1, J2 = params.J1, params.J2
    cc, c1, c2, g = params.cc, params.c1, params.c2, params.g
    x1, x2, x3, x4, x5, x6 = [state[..., i] for i in range(6)]
    return (m2*(l2**2*x6**2+x2**2+L1**2*x4**2-L1*x2*x4*xp.cos(x3)*2.0-l2*x2*x6*xp.cos(x5)*2.0+L1*l2*x4*x6*xp.cos(x3-x5)*2.0))/2.0+(J1*x4**2)/2.0+(J2*x6**2)/2.0+g*(m2*(L1*xp.cos(x3)+l2*xp.cos(x5))+l1*m1*xp.cos(x3))+(m1*(l1**2*x4**2+x2**2-l1*x2*x4*xp.cos(x3)*2.0))/2.0+(mc*x2**2)/2.0

def desired_upright_energy(params):
    return compute_energy(np.zeros(6), params)

EXPERT_UPRIGHT_ENERGY = float(desired_upright_energy(EXPERT_PARAMS))

ENERGY_CASES = {
    "down-down": np.array([0, 0, -np.pi, 0, -np.pi, 0], dtype=float),
    "up-up": np.zeros(6),
    "small velocity": np.array([0, 0.1, 0, 0.1, 0, -0.1]),
    "nonzero velocity": np.array([0, 1.0, 0.4, 2.0, -0.3, -1.0]),
}
for name, test_state in ENERGY_CASES.items():
    value = float(compute_energy(test_state, EXPERT_PARAMS))
    assert np.isfinite(value)
    print(f"{name:>16}: {value: .6f} J")
assert np.isclose(compute_energy(np.zeros(6), EXPERT_PARAMS), desired_upright_energy(EXPERT_PARAMS))


## SECTION 10 — Nonlinear swing-up controller

The repository law is preserved: remove cart kinetic energy, compute coupling `W`, form `v`, clip `v` to ±75, then apply `u = alpha + beta*v`. V2 additionally clips final force to the actual MuJoCo actuator range, fixing the reference implementation's unbounded final force.


In [ ]:
def swingup_virtual_control(state, params, config=EXPERT_CONFIG, xp=np):
    pendulum_state = xp.stack(
        [state[..., 0], xp.zeros_like(state[..., 1]), *[state[..., i] for i in range(2, 6)]],
        axis=-1,
    )
    energy_error = compute_energy(pendulum_state, params, xp) - EXPERT_UPRIGHT_ENERGY
    W = (
        (params.m1 * params.l1 + params.m2 * params.L1) * state[..., 3] * xp.cos(state[..., 2])
        + params.m2 * params.l2 * state[..., 5] * xp.cos(state[..., 4])
    )
    v = (
        -config["energy_gain"] * energy_error * W
        -config["cart_kp"] * state[..., 0]
        -config["cart_kd"] * state[..., 1]
    )
    return xp.clip(v, -config["virtual_force_limit"], config["virtual_force_limit"])

def swingup_force(state, params=EXPERT_PARAMS, config=EXPERT_CONFIG, xp=np):
    alpha, beta = compute_alpha_beta(state, params, xp)
    force = alpha + beta * swingup_virtual_control(state, params, config, xp)
    return xp.clip(force, -ACTION_LIMIT, ACTION_LIMIT)


## SECTION 11 — Linear stabilizer

LQR is used instead of porting the reference QP. The **actual MuJoCo one-step map** is numerically linearized in the canonical physical coordinates at up-up. The discrete Riccati equation produces `K`; closed-loop eigenvalues must lie inside the unit circle.


In [ ]:
def _cpu_discrete_dynamics(state, force):
    z = physical_to_mujoco_state(np.asarray(state), np)
    data = mujoco.MjData(mj_model)
    data.qpos[:] = z[:3]
    data.qvel[:] = z[3:]
    data.ctrl[0] = force
    mujoco.mj_forward(mj_model, data)
    mujoco.mj_step(mj_model, data)
    x, x_dot, theta1, theta1_dot, theta2, theta2_dot = physical_state(data)
    return np.array([x, x_dot, theta1, theta1_dot, theta2, theta2_dot])

def compute_upright_lqr_gain(eps_state=1e-5, eps_u=1e-4):
    equilibrium = np.zeros(6)
    A = np.column_stack([
        (_cpu_discrete_dynamics(equilibrium + np.eye(6)[i] * eps_state, 0.0)
         - _cpu_discrete_dynamics(equilibrium - np.eye(6)[i] * eps_state, 0.0))
        / (2.0 * eps_state)
        for i in range(6)
    ])
    B = ((_cpu_discrete_dynamics(equilibrium, eps_u)
          - _cpu_discrete_dynamics(equilibrium, -eps_u)) / (2.0 * eps_u))[:, None]
    Q = np.diag(EXPERT_CONFIG["lqr_q"])
    R = np.array([[EXPERT_CONFIG["lqr_r"]]])
    P = solve_discrete_are(A, B, Q, R)
    K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
    eigenvalues = np.linalg.eigvals(A - B @ K)
    assert np.max(np.abs(eigenvalues)) < 1.0
    return A, B, K.astype(np.float32), eigenvalues

LQR_A, LQR_B, EXPERT_LQR_K_NP, LQR_EIGENVALUES = compute_upright_lqr_gain()
EXPERT_LQR_K = jnp.asarray(EXPERT_LQR_K_NP)
print("LQR K:", EXPERT_LQR_K_NP)
print("closed-loop eigenvalues:", LQR_EIGENVALUES)

def lqr_action(state):
    state = np.asarray(state, dtype=float).copy()
    state[2] = float(wrap_angle(state[2]))
    state[4] = float(wrap_angle(state[4]))
    return float(np.clip(-(EXPERT_LQR_K_NP @ state)[0], -ACTION_LIMIT, ACTION_LIMIT))


## SECTION 12 — Hybrid Teacher

The readable CPU teacher switches from the converted nonlinear swing-up law to MuJoCo-specific LQR using the source controller's capture condition: both wrapped angles within 15° and `|theta2_dot| < 2 rad/s`. Capture is latched for the rollout.


In [ ]:
class Teacher:
    SWING_UP = 0
    STABILIZE = 1

    def __init__(self, params=EXPERT_PARAMS, config=EXPERT_CONFIG):
        self.params = params
        self.config = config
        self.latched = False

    def reset(self):
        self.latched = False

    def swing_up(self, state):
        return float(swingup_force(np.asarray(state), self.params, self.config, np))

    def stabilize(self, state):
        return lqr_action(state)

    def inside_capture_region(self, state):
        x, x_dot, theta1, theta1_dot, theta2, theta2_dot = np.asarray(state)
        return (
            abs(float(wrap_angle(theta1))) < self.config["capture_angle_rad"]
            and abs(float(wrap_angle(theta2))) < self.config["capture_angle_rad"]
            and abs(theta1_dot) < self.config["capture_rate_rad_s"]
            and abs(theta2_dot) < self.config["capture_rate_rad_s"]
        )

    def action(self, state):
        self.latched = self.latched or self.inside_capture_region(state)
        mode = self.STABILIZE if self.latched else self.SWING_UP
        force = self.stabilize(state) if self.latched else self.swing_up(state)
        return force, mode


## SECTION 13 — Expert single-environment validation

This gate runs before imitation. It logs time, all six physical states, energy, alpha, beta, force, and controller mode, then creates six diagnostic plots. Down-down starts with a small angular velocity perturbation (theta1_dot=theta2_dot=0.05 rad/s) to activate the energy-pumping controller, plus matched capture thresholds (30 deg, 4 rad/s). If the expert never reaches upright, execution stops before dataset generation.


In [ ]:
def cpu_reward(state, normalized_action):
    x, x_dot, theta1, theta1_dot, theta2, theta2_dot = state
    upright = abs(theta1) < 0.35 and abs(theta2) < 0.35
    return (
        0.5 * (np.cos(theta1) + np.cos(theta2))
        - 0.25 * (x / RAIL_LIMIT) ** 2
        - 0.01 * x_dot**2
        - 0.003 * (theta1_dot**2 + theta2_dot**2)
        - 0.001 * normalized_action**2
        + 3.0 * upright
        - 100.0 * (abs(x) >= RAIL_LIMIT)
    )

def run_cpu_teacher(initial_state, max_steps=MAX_EPISODE_STEPS):
    data = mujoco.MjData(mj_model)
    z = physical_to_mujoco_state(np.asarray(initial_state), np)
    data.qpos[:] = z[:3]
    data.qvel[:] = z[3:]
    mujoco.mj_forward(mj_model, data)
    teacher = Teacher()
    records = []
    captured_once = False
    for step in range(max_steps):
        state = np.asarray(physical_state_vector(data), dtype=float)
        alpha, beta = compute_alpha_beta(state, EXPERT_PARAMS)
        force, mode = teacher.action(state)
        normalized_action = force / ACTION_LIMIT
        energy_state = state.copy(); energy_state[1] = 0.0
        records.append({
            "time": step * mj_model.opt.timestep,
            "x": state[0], "x_dot": state[1],
            "theta1": state[2], "theta1_dot": state[3],
            "theta2": state[4], "theta2_dot": state[5],
            "energy": compute_energy(energy_state, EXPERT_PARAMS),
            "alpha": alpha, "beta": beta, "force": force, "mode": mode,
            "reward": cpu_reward(state, normalized_action),
        })
        captured_once = captured_once or (abs(state[2]) < 0.35 and abs(state[4]) < 0.35)
        data.ctrl[0] = force
        mujoco.mj_step(mj_model, data)
        if abs(data.qpos[0]) >= RAIL_LIMIT:
            break
    return pd.DataFrame(records), captured_once

showcase_initial_state = np.array([0.0, 0.0, -np.pi, 0.05, -np.pi, 0.05])
EXPERT_VALIDATED = False
if RUN_CPU_EXPERT_VALIDATION:
    expert_trace, EXPERT_VALIDATED = run_cpu_teacher(showcase_initial_state)
    fig, axes = plt.subplots(6, 1, figsize=(11, 16), sharex=True)
    axes[0].plot(expert_trace.time, expert_trace.x); axes[0].set_ylabel("x [m]")
    axes[1].plot(expert_trace.time, expert_trace[["theta1", "theta2"]]); axes[1].set_ylabel("angle [rad]")
    axes[2].plot(expert_trace.time, expert_trace[["theta1_dot", "theta2_dot"]]); axes[2].set_ylabel("rate [rad/s]")
    axes[3].plot(expert_trace.time, expert_trace.force); axes[3].set_ylabel("force [N]")
    axes[4].plot(expert_trace.time, expert_trace.energy); axes[4].axhline(desired_upright_energy(EXPERT_PARAMS), ls="--"); axes[4].set_ylabel("energy [J]")
    axes[5].step(expert_trace.time, expert_trace["mode"], where="post"); axes[5].set_ylabel("mode"); axes[5].set_xlabel("time [s]")
    fig.tight_layout(); fig.savefig(PLOT_DIR / "expert_validation.png", dpi=180)
    plt.show()
    if not EXPERT_VALIDATED:
        raise RuntimeError("Expert failed the independent down-down validation; do not generate imitation data.")


## SECTION 14 — Expert video

The validated CPU teacher is rendered with the unchanged XML before imitation. Output: `outputs/videos/expert_swingup_stabilization.mp4`.


In [ ]:
def render_cpu_policy(path, initial_state, policy, seconds=RENDER_CONFIG["seconds"]):
    model = mujoco.MjModel.from_xml_string(MODEL_XML)
    data = mujoco.MjData(model)
    z = physical_to_mujoco_state(np.asarray(initial_state), np)
    data.qpos[:] = z[:3]; data.qvel[:] = z[3:]
    mujoco.mj_forward(model, data)
    renderer = mujoco.Renderer(model, RENDER_CONFIG["height"], RENDER_CONFIG["width"])
    frames = []
    frame_stride = max(1, round(1.0 / (RENDER_CONFIG["fps"] * model.opt.timestep)))
    for step in range(round(seconds / model.opt.timestep)):
        state = np.asarray(physical_state_vector(data), dtype=float)
        force, mode = policy(data, state)
        data.ctrl[0] = np.clip(force, -ACTION_LIMIT, ACTION_LIMIT)
        mujoco.mj_step(model, data)
        if step % frame_stride == 0:
            renderer.update_scene(data, camera=RENDER_CONFIG["camera"])
            image = Image.fromarray(renderer.render())
            ImageDraw.Draw(image).text(
                (12, 12),
                f"t={step * model.opt.timestep:5.2f}s  mode={mode}\nx={state[0]:+.2f}  theta=({state[2]:+.2f}, {state[4]:+.2f})  F={force:+.1f}N",
                fill="white",
                stroke_width=2,
                stroke_fill="black",
            )
            frames.append(np.asarray(image))
    renderer.close()
    imageio.mimsave(path, frames, fps=RENDER_CONFIG["fps"], codec="libx264")
    return path

expert_video_path = VIDEO_DIR / "expert_swingup_stabilization.mp4"
if EXPERT_VALIDATED:
    video_teacher = Teacher()
    render_cpu_policy(
        expert_video_path,
        showcase_initial_state,
        lambda data, state: video_teacher.action(state),
    )
    display(Video(str(expert_video_path), embed=True))


## SECTION 15 — Expert metrics

Randomized down-down initial states are evaluated with common metric definitions. Metrics are retained for the final paired comparison.


In [ ]:
def summarize_trace(trace, captured):
    capture_rows = trace[(trace.theta1.abs() < 0.35) & (trace.theta2.abs() < 0.35)]
    return {
        "success": float(captured and len(capture_rows) > 0),
        "return": float(trace.reward.sum()),
        "capture_time": float(capture_rows.time.iloc[0]) if len(capture_rows) else np.nan,
        "max_cart_displacement": float(trace.x.abs().max()),
        "cart_rms": float(np.sqrt(np.mean(trace.x**2))),
        "control_effort": float(np.sum(trace.force**2) * mj_model.opt.timestep),
        "reached_upright": float(captured),
    }

def save_metric_summary(name, rows):
    frame = pd.DataFrame(rows)
    frame.to_csv(METRIC_DIR / f"{name}_episodes.csv", index=False)
    summary = frame.mean(numeric_only=True).to_dict()
    (METRIC_DIR / f"{name}_metrics.json").write_text(json.dumps(summary, indent=2))
    return summary

def evaluate_cpu_teacher(seeds):
    rows = []
    for seed in seeds:
        random = np.random.default_rng(seed)
        initial = showcase_initial_state + random.uniform(
            [-0.01, -0.01, -0.04, -0.02, -0.04, -0.02],
            [0.01, 0.01, 0.04, 0.02, 0.04, 0.02],
        )
        trace, captured = run_cpu_teacher(initial)
        rows.append(summarize_trace(trace, captured))
    return rows

COMPARISON_SEEDS = np.arange(SEED, SEED + EXPERT_CONFIG["evaluation_episodes"])
expert_episode_metrics = evaluate_cpu_teacher(COMPARISON_SEEDS) if EXPERT_VALIDATED else []
EXPERT_METRICS = save_metric_summary("expert", expert_episode_metrics) if expert_episode_metrics else {}
print(EXPERT_METRICS)


# PART III — GPU EXPERT / DATASET

## SECTION 16 — JAX-native expert

The same equations, LQR gain, clipping, and capture test are functional JAX. Capture latch is explicit scan state. No Python, NumPy, SciPy, repository access, or file loading executes inside the jitted rollout.


In [ ]:
def jax_teacher_action(data, latched):
    state = physical_state_vector(data)
    theta1 = wrap_angle(state[..., 2])
    theta2 = wrap_angle(state[..., 4])
    capture = (
        (jnp.abs(theta1) < EXPERT_CONFIG["capture_angle_rad"])
        & (jnp.abs(theta2) < EXPERT_CONFIG["capture_angle_rad"])
        & (jnp.abs(state[..., 3]) < EXPERT_CONFIG["capture_rate_rad_s"])
        & (jnp.abs(state[..., 5]) < EXPERT_CONFIG["capture_rate_rad_s"])
    )
    latched = latched | capture
    swing_force = swingup_force(state, EXPERT_PARAMS, EXPERT_CONFIG, jnp)
    wrapped_state = state.at[..., 2].set(theta1).at[..., 4].set(theta2)
    lqr_force = -jnp.einsum("ij,...j->...i", EXPERT_LQR_K, wrapped_state)[..., 0]
    force = jnp.clip(jnp.where(latched, lqr_force, swing_force), -ACTION_LIMIT, ACTION_LIMIT)
    return (force / ACTION_LIMIT)[..., None].astype(jnp.float32), latched

jax_teacher_action = jax.jit(jax_teacher_action)

comparison_states = np.stack([
    showcase_initial_state,
    np.zeros(6),
    np.array([0.2, -0.1, 0.1, 0.2, -0.08, -0.1]),
])
for sample in comparison_states:
    data = mujoco.MjData(mj_model)
    z = physical_to_mujoco_state(sample, np); data.qpos[:] = z[:3]; data.qvel[:] = z[3:]
    mujoco.mj_forward(mj_model, data)
    cpu_teacher = Teacher(); cpu_force, _ = cpu_teacher.action(sample)
    mjx_data = mjx.put_data(mj_model, data)
    jax_action, _ = jax_teacher_action(mjx_data, jnp.asarray(False))
    jax_force = float(jax_action[0] * ACTION_LIMIT)
    np.testing.assert_allclose(cpu_force, jax_force, rtol=2e-4, atol=2e-3)
    print("CPU Teacher action vs JAX Teacher action:", cpu_force, jax_force)


## SECTION 17 — Batched MJX expert rollout

One compiled `NUM_ENVS × ROLLOUT_STEPS` collection stores observation, normalized expert action, done, canonical physical state, expert mode, and episode capture. Arrays remain on device.


In [ ]:
@struct.dataclass
class ExpertTransition:
    obs: jax.Array
    action: jax.Array
    done: jax.Array
    physical: jax.Array
    mode: jax.Array
    captured: jax.Array

def collect_expert_rollout_impl(env_state, obs, rng, latched):
    def one_step(carry, _):
        env_state, obs, rng, latched = carry
        rng, reset_key = jax.random.split(rng)
        action, next_latched = jax_teacher_action(env_state.data, latched)
        assert action.shape == (NUM_ENVS, ACTION_DIM)
        assert action.dtype == jnp.float32
        stepped_data, terminal_obs, reward, terminated, truncated, done, episode_return, episode_length, captured = step_batch(env_state, action)
        reset_data = reset_data_batch(jax.random.split(reset_key, NUM_ENVS))
        next_data = jax.vmap(
            lambda stepped, reset, is_done: stepped.where(is_done, reset)
        )(stepped_data, reset_data, done)
        next_obs = jnp.where(done[:, None], observation_from_data(reset_data), terminal_obs)
        next_state = EnvState(next_data, jnp.where(done, 0, env_state.step_count + 1), jnp.where(done, 0.0, episode_return), jnp.where(done, 0, episode_length), jnp.where(done, False, captured))
        transition = ExpertTransition(
            obs=obs,
            action=action,
            done=done,
            physical=physical_state_vector(env_state.data),
            mode=next_latched.astype(jnp.int8),
            captured=captured,
        )
        return (next_state, next_obs, rng, jnp.where(done, False, next_latched)), transition
    carry, trajectory = jax.lax.scan(one_step, (env_state, obs, rng, latched), None, length=ROLLOUT_STEPS)
    return (*carry, trajectory)

collect_expert_rollout = jax.jit(collect_expert_rollout_impl)

def flatten_expert_trajectory(expert_trajectory):
    """Flatten [time, environment, ...] axes with explicit field shapes."""
    return ExpertTransition(
        obs=expert_trajectory.obs.reshape((-1, OBS_DIM)),
        action=expert_trajectory.action.reshape((-1, ACTION_DIM)),
        done=expert_trajectory.done.reshape(-1),
        physical=expert_trajectory.physical.reshape((-1, 6)),
        mode=expert_trajectory.mode.reshape(-1),
        captured=expert_trajectory.captured.reshape(-1),
    )
rng = jax.random.PRNGKey(SEED + 1000)
rng, expert_reset_key = jax.random.split(rng)
expert_env_state, expert_obs = reset_batch(jax.random.split(expert_reset_key, NUM_ENVS))
expert_latched = jnp.zeros(NUM_ENVS, dtype=bool)
EXPERT_DIAGNOSTIC_CHUNKS = (MAX_EPISODE_STEPS + ROLLOUT_STEPS - 1) // ROLLOUT_STEPS
diagnostic_parts = {"action": [], "physical": [], "mode": [], "captured": []}
for diagnostic_chunk in range(EXPERT_DIAGNOSTIC_CHUNKS):
    expert_env_state, expert_obs, rng, expert_latched, EXPERT_TRAJECTORY = collect_expert_rollout(
        expert_env_state, expert_obs, rng, expert_latched
    )
    EXPERT_DATASET = flatten_expert_trajectory(EXPERT_TRAJECTORY)
    diagnostic_stride = max(1, BC_BATCH_SIZE // 2048)
    for name in diagnostic_parts:
        diagnostic_parts[name].append(
            np.asarray(jax.device_get(getattr(EXPERT_DATASET, name)[::diagnostic_stride]))
        )
EXPERT_DIAGNOSTIC_SAMPLE = {
    name: np.concatenate(parts, axis=0) for name, parts in diagnostic_parts.items()
}
print("expert dataset:", EXPERT_DATASET.obs.shape, EXPERT_DATASET.action.shape)


## SECTION 18 — Demonstration diagnostics

Histograms and proportions sample at least one maximum-episode horizon before BC. Only bounded diagnostic samples cross to CPU; full training batches remain device-resident. Arrays are normalized to explicit NumPy ranks before plotting, preventing tuple/action-shape ambiguity. A zero stabilization proportion stops execution before BC.


In [ ]:
diag_action = np.asarray(EXPERT_DIAGNOSTIC_SAMPLE["action"], dtype=np.float32).reshape(-1)
diag_state = np.asarray(EXPERT_DIAGNOSTIC_SAMPLE["physical"], dtype=np.float32).reshape(-1, 6)
diag_mode = np.asarray(EXPERT_DIAGNOSTIC_SAMPLE["mode"], dtype=np.int8).reshape(-1)
diag_captured = np.asarray(EXPERT_DIAGNOSTIC_SAMPLE["captured"], dtype=bool).reshape(-1)
assert diag_action.ndim == 1
assert diag_state.ndim == 2 and diag_state.shape[1] == 6
assert diag_mode.shape == diag_captured.shape == diag_action.shape
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(11, 8))
axes[0, 0].hist(diag_action, bins=int(50)); axes[0, 0].set_title("normalized expert action")
axes[0, 1].hist(diag_state[:, 2], bins=50, alpha=.7, label="theta1"); axes[0, 1].hist(diag_state[:, 4], bins=50, alpha=.7, label="theta2"); axes[0, 1].legend()
axes[1, 0].hist(diag_state[:, 0], bins=50); axes[1, 0].set_title("cart position")
axes[1, 1].bar(["swing-up", "stabilize"], [np.mean(diag_mode == 0), np.mean(diag_mode == 1)])
fig.tight_layout(); fig.savefig(PLOT_DIR / "demonstration_diagnostics.png", dpi=180); plt.show()
print("swing-up proportion:", np.mean(diag_mode == 0))
print("stabilization proportion:", np.mean(diag_mode == 1))
print("expert captured percentage:", 100 * float(np.mean(diag_captured)))
if not np.any(diag_mode == Teacher.STABILIZE):
    raise RuntimeError("Expert demonstrations contain no stabilization states; stop before BC.")


# PART IV — IMITATION LEARNING

## SECTION 19 — Policy network

This is the unchanged shared actor-critic used by PPO: two 256-unit tanh layers, tanh-squashed Gaussian actor, learned log standard deviation, and value head. BC and PPO therefore share parameters directly.


In [ ]:
class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, obs):
        x = nn.tanh(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(obs))
        x = nn.tanh(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(x))

        mean = nn.Dense(
            self.action_dim,
            kernel_init=nn.initializers.orthogonal(0.01),
            bias_init=nn.initializers.zeros_init(),
        )(x)

        log_std = self.param(
            "log_std",
            nn.initializers.constant(-1.5),
            (self.action_dim,),
        )
        log_std = jnp.clip(log_std, -4.0, 0.0)

        value = nn.Dense(
            1,
            kernel_init=nn.initializers.orthogonal(1.0),
            bias_init=nn.initializers.zeros_init(),
        )(x)

        return mean, log_std, jnp.squeeze(value, axis=-1)

network = ActorCritic(ACTION_DIM)

def gaussian_log_prob(raw_action, mean, log_std):
    std = jnp.exp(log_std)
    z = (raw_action - mean) / std
    return -0.5 * jnp.sum(
        z**2 + 2.0 * log_std + jnp.log(2.0 * jnp.pi),
        axis=-1,
    )

def squashed_log_prob(raw_action, action, mean, log_std):
    base_log_prob = gaussian_log_prob(raw_action, mean, log_std)
    correction = jnp.sum(
        jnp.log(jnp.clip(1.0 - action**2, 1e-6, 1.0)),
        axis=-1,
    )
    return base_log_prob - correction

def gaussian_entropy(log_std):
    return jnp.sum(log_std + 0.5 * jnp.log(2.0 * jnp.pi * jnp.e))

def sample_action(params, obs, key):
    mean, log_std, value = network.apply(params, obs)
    noise = jax.random.normal(key, mean.shape)
    raw_action = mean + jnp.exp(log_std) * noise
    action = jnp.tanh(raw_action)
    log_prob = squashed_log_prob(raw_action, action, mean, log_std)
    return raw_action, action, log_prob, value

def greedy_action(params, obs):
    mean, _, value = network.apply(params, obs)
    return jnp.tanh(mean), value


## SECTION 20 — BC loss

Expert targets are normalized actions. PPO predicts a Gaussian mean in pre-tanh space, so BC transforms clipped targets with `arctanh` and minimizes latent MSE. Action MAE is reported after tanh. The value head receives no BC loss.


In [ ]:
class TrainState(train_state.TrainState):
    pass

def bc_metrics(params, batch):
    mean, log_std, value = network.apply(params, batch["obs"])
    target_action = jnp.clip(batch["action"], -0.999, 0.999)
    target_raw = jnp.arctanh(target_action)
    loss = jnp.mean((mean - target_raw) ** 2)
    mae = jnp.mean(jnp.abs(jnp.tanh(mean) - target_action))
    return loss, {"bc_loss": loss, "action_mae": mae}

def _bc_minibatch_update(state, batch):
    (loss, metrics), grads = jax.value_and_grad(bc_metrics, has_aux=True)(state.params, batch)
    return state.apply_gradients(grads=grads), metrics

def _bc_update_epochs(state, batch, rng):
    size = batch["obs"].shape[0]
    minibatch_size = size // BC_NUM_MINIBATCHES
    def epoch_step(carry, _):
        state, rng = carry
        rng, key = jax.random.split(rng)
        indices = jax.random.permutation(key, size)[:minibatch_size * BC_NUM_MINIBATCHES]
        indices = indices.reshape(BC_NUM_MINIBATCHES, minibatch_size)
        def minibatch_step(state, index):
            return _bc_minibatch_update(state, jax.tree.map(lambda x: x[index], batch))
        state, metrics = jax.lax.scan(minibatch_step, state, indices)
        return (state, rng), metrics
    (state, rng), metrics = jax.lax.scan(epoch_step, (state, rng), None, length=BC_EPOCHS)
    return state, rng, jax.tree.map(jnp.mean, metrics)

bc_update_epochs = jax.jit(_bc_update_epochs)


## SECTION 21 — BC training

Training uses only fresh device-resident expert batches while continuing the full-horizon expert environments from Section 17. Every batch receives a deterministic 90/10 split. This preserves the working online-collection pattern while exposing swing-up and stabilization states. It logs train/validation MSE and action MAE and saves `outputs/checkpoints/bc_pretrained.pkl`.


In [ ]:
rng, init_key = jax.random.split(rng)
initial_params = network.init(init_key, jnp.zeros((1, OBS_DIM), dtype=jnp.float32))
bc_optimizer = optax.chain(optax.clip_by_global_norm(MAX_GRAD_NORM), optax.adam(BC_LEARNING_RATE))
bc_state = TrainState.create(apply_fn=network.apply, params=initial_params, tx=bc_optimizer)
BC_HISTORY = []
if RUN_BC_TRAINING:
    for update in range(1, BC_UPDATES + 1):
        expert_env_state, expert_obs, rng, expert_latched, expert_trajectory = collect_expert_rollout(
            expert_env_state, expert_obs, rng, expert_latched
        )
        expert_batch = flatten_expert_trajectory(expert_trajectory)
        rng, split_key = jax.random.split(rng)
        split = int(BC_BATCH_SIZE * (1.0 - BC_CONFIG["validation_fraction"]))
        permutation = jax.random.permutation(split_key, BC_BATCH_SIZE)
        train_index, validation_index = permutation[:split], permutation[split:]
        bc_train_batch = {"obs": expert_batch.obs[train_index], "action": expert_batch.action[train_index]}
        bc_validation_batch = {"obs": expert_batch.obs[validation_index], "action": expert_batch.action[validation_index]}
        bc_state, rng, train_metrics = bc_update_epochs(bc_state, bc_train_batch, rng)
        validation_loss, validation_metrics = bc_metrics(bc_state.params, bc_validation_batch)
        row = {
            "update": update,
            "train_loss": float(train_metrics["bc_loss"]),
            "train_mae": float(train_metrics["action_mae"]),
            "validation_loss": float(validation_loss),
            "validation_mae": float(validation_metrics["action_mae"]),
        }
        BC_HISTORY.append(row)
        if update == 1 or update % 5 == 0 or update == BC_UPDATES:
            print(row)
    with open(BC_CHECKPOINT_PATH, "wb") as file:
        pickle.dump({"params": jax.device_get(bc_state.params), "history": BC_HISTORY}, file, pickle.HIGHEST_PROTOCOL)
    pd.DataFrame(BC_HISTORY).to_csv(METRIC_DIR / "bc_training.csv", index=False)
else:
    with open(BC_CHECKPOINT_PATH, "rb") as file:
        bc_payload = pickle.load(file)
    bc_state = bc_state.replace(params=jax.device_put(bc_payload["params"]))
    BC_HISTORY = bc_payload.get("history", [])


## SECTION 22 — BC evaluation

Held-out MSE is insufficient. The deterministic BC policy is evaluated in the real CPU MuJoCo plant over the same seeds and metric definitions as the expert.


In [ ]:
def run_cpu_neural_policy(params, initial_state, max_steps=MAX_EPISODE_STEPS):
    data = mujoco.MjData(mj_model)
    z = physical_to_mujoco_state(np.asarray(initial_state), np)
    data.qpos[:] = z[:3]; data.qvel[:] = z[3:]
    mujoco.mj_forward(mj_model, data)
    records, captured = [], False
    for step in range(max_steps):
        state = np.asarray(physical_state_vector(data), dtype=float)
        obs = np.asarray(observation_from_data(data), dtype=np.float32)[None]
        action = float(greedy_action(params, jnp.asarray(obs))[0, 0])
        force = action * ACTION_LIMIT
        upright = abs(state[2]) < 0.35 and abs(state[4]) < 0.35
        captured = captured or upright
        records.append({
            "time": step * mj_model.opt.timestep, "x": state[0], "x_dot": state[1],
            "theta1": state[2], "theta1_dot": state[3], "theta2": state[4], "theta2_dot": state[5],
            "force": force, "reward": cpu_reward(state, action), "mode": "NEURAL",
        })
        data.ctrl[0] = force; mujoco.mj_step(mj_model, data)
        if abs(data.qpos[0]) >= RAIL_LIMIT: break
    return pd.DataFrame(records), captured

def evaluate_neural_policy(params, seeds, name):
    rows = []
    for seed in seeds:
        random = np.random.default_rng(seed)
        initial = showcase_initial_state + random.uniform(
            [-0.01, -0.01, -0.04, -0.02, -0.04, -0.02],
            [0.01, 0.01, 0.04, 0.02, 0.04, 0.02],
        )
        trace, captured = run_cpu_neural_policy(params, initial)
        rows.append(summarize_trace(trace, captured))
    return rows, save_metric_summary(name, rows)

bc_episode_metrics, BC_METRICS = evaluate_neural_policy(bc_state.params, COMPARISON_SEEDS, "bc")
print(BC_METRICS)


## SECTION 23 — BC video

The BC video uses the same showcase initial condition as the expert. Output: `outputs/videos/bc_policy.mp4`.


In [ ]:
bc_video_path = VIDEO_DIR / "bc_policy.mp4"
def bc_video_policy(data, state):
    obs = observation_from_data(data)[None]
    action = float(greedy_action(bc_state.params, obs)[0, 0])
    return action * ACTION_LIMIT, "BC"
render_cpu_policy(bc_video_path, showcase_initial_state, bc_video_policy)
display(Video(str(bc_video_path), embed=True))


# PART V — PPO FINE-TUNING

## SECTION 24 — PPO configuration

All PPO hyperparameters live in `PPO_CONFIG` in Section 2. Values are preserved from the working notebook: 2048 environments, 64-step rollouts, 300M transitions, γ=.99, λ=.95, clip=.2, six epochs, 16 minibatches, entropy coefficient .0001, value coefficient .5, and gradient norm .5.


## SECTION 25 — Initialize PPO from BC

For a new run, transfer is explicit: copy BC network parameters, discard the BC optimizer state, and create a fresh PPO Adam optimizer. Shared hidden layers and actor are retained; the BC-untrained critic head remains at its original initialization because its gradients were zero during BC. If `PPO_CONFIG["resume"]` is enabled and `ppo_latest.pkl` exists, the preserved checkpoint restoration path resumes that PPO run instead.


In [ ]:
import csv
import pickle
import time
from datetime import datetime, timezone

METRIC_FIELDS = [
    "timestamp",
    "update",
    "total_steps",
    "reward",
    "policy_loss",
    "value_loss",
    "entropy",
    "approx_kl",
    "clip_fraction",
    "episode_length",
    "steps_per_second",
    "both_upright",
    "episodes_finished",
]

def save_checkpoint(train_state, rng, update, total_steps):
    payload = {
        "version": 2,
        "update": int(update),
        "total_steps": int(total_steps),
        "params": jax.device_get(train_state.params),
        "opt_state": jax.device_get(train_state.opt_state),
        "rng": np.asarray(jax.device_get(rng)),
    }

    final_path = CHECKPOINT_DIR / f"checkpoint_{total_steps:012d}.pkl"
    tmp_path = final_path.with_suffix(".tmp")

    with open(tmp_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, final_path)

    latest = PPO_LATEST_PATH
    latest_tmp = CHECKPOINT_DIR / "latest.tmp"
    with open(latest_tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(latest_tmp, latest)

    return final_path

def restore_checkpoint(train_state):
    latest = PPO_LATEST_PATH
    if not latest.exists():
        return train_state, jax.random.PRNGKey(SEED), 0, 0

    with open(latest, "rb") as f:
        payload = pickle.load(f)

    train_state = train_state.replace(
        params=jax.device_put(payload["params"]),
        opt_state=jax.device_put(payload["opt_state"]),
    )

    restored_rng = jnp.asarray(payload["rng"], dtype=jnp.uint32)
    print(
        "Restored:",
        "update", payload["update"],
        "| steps", payload["total_steps"],
    )
    return (
        train_state,
        restored_rng,
        int(payload["update"]),
        int(payload["total_steps"]),
    )

def append_metric(row):
    exists = METRICS_PATH.exists()
    with open(METRICS_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=METRIC_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def make_ppo_train_state(params):
    optimizer = optax.chain(optax.clip_by_global_norm(MAX_GRAD_NORM), optax.adam(LEARNING_RATE))
    return TrainState.create(apply_fn=network.apply, params=params, tx=optimizer)

ppo_initial_params = jax.tree.map(lambda x: x.copy(), bc_state.params)
state = make_ppo_train_state(ppo_initial_params)  # fresh optimizer; no BC Adam moments
rng = jax.random.PRNGKey(SEED + 2000)
update_index = 0
total_steps = 0
if PPO_CONFIG["resume"] and PPO_LATEST_PATH.exists():
    state, rng, update_index, total_steps = restore_checkpoint(state)

def atomic_copy(source, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_bytes(source.read_bytes())
    os.replace(temporary, destination)


## SECTION 26 — PPO rollout

The working GPU rollout is preserved: `jax.jit`, `jax.vmap`, `jax.lax.scan`, parallel MJX environments, tanh-squashed actions, terminal-observation values, and device-resident transitions.


In [ ]:
@struct.dataclass
class Transition:
    obs: jax.Array
    raw_action: jax.Array
    action: jax.Array
    log_prob: jax.Array
    value: jax.Array
    reward: jax.Array
    terminated: jax.Array
    truncated: jax.Array
    done: jax.Array
    next_value: jax.Array
    completed_return: jax.Array
    completed_length: jax.Array
    completed_capture: jax.Array

def _collect_rollout(train_state, env_state, obs, rng):
    def one_step(carry, _):
        env_state, obs, rng = carry
        rng, action_key, reset_key = jax.random.split(rng, 3)

        raw_action, action, log_prob, value = sample_action(
            train_state.params, obs, action_key
        )

        (
            stepped_data,
            terminal_obs,
            reward,
            terminated,
            truncated,
            done,
            episode_return,
            episode_length,
            captured,
        ) = step_batch(env_state, action)

        # Value of the true post-step state BEFORE any reset.
        _, _, terminal_value = network.apply(
            train_state.params, terminal_obs
        )

        reset_keys = jax.random.split(reset_key, NUM_ENVS)
        reset_data = reset_data_batch(reset_keys)

        # Select reset state per environment.
        #
        # Data.where() with a vector mask does not broadcast correctly over
        # leaves such as qpos=(NUM_ENVS, 3).  Instead, vmap Data.where so each
        # environment receives a scalar boolean mask.
        next_data = jax.vmap(
            lambda stepped, reset, is_done: stepped.where(is_done, reset)
        )(stepped_data, reset_data, done)

        reset_obs = observation_from_data(reset_data)
        next_obs = jnp.where(done[:, None], reset_obs, terminal_obs)

        completed_return = jnp.where(done, episode_return, jnp.nan)
        completed_length = jnp.where(done, episode_length, 0)
        completed_capture = jnp.where(done, captured, False)

        next_env_state = EnvState(
            data=next_data,
            step_count=jnp.where(done, 0, env_state.step_count + 1),
            episode_return=jnp.where(done, 0.0, episode_return),
            episode_length=jnp.where(done, 0, episode_length),
            captured_upright=jnp.where(done, False, captured),
        )

        transition = Transition(
            obs=obs,
            raw_action=raw_action,
            action=action,
            log_prob=log_prob,
            value=value,
            reward=reward,
            terminated=terminated,
            truncated=truncated,
            done=done,
            next_value=terminal_value,
            completed_return=completed_return,
            completed_length=completed_length,
            completed_capture=completed_capture,
        )

        return (next_env_state, next_obs, rng), transition

    (env_state, obs, rng), traj = jax.lax.scan(
        one_step,
        (env_state, obs, rng),
        xs=None,
        length=ROLLOUT_STEPS,
    )

    return env_state, obs, rng, traj

collect_rollout = jax.jit(_collect_rollout)


## SECTION 27 — GAE

The working GAE is preserved. True rail terminals do not bootstrap; time-limit truncations bootstrap from terminal observations; both stop recursion so episodes are not stitched.


In [ ]:
@jax.jit
def compute_gae(traj):
    def reverse_step(gae, t):
        # Bootstrap at ordinary transitions and at time-limit truncations,
        # but never across a true terminal rail failure.
        bootstrap_mask = 1.0 - t.terminated.astype(jnp.float32)
        recursion_mask = 1.0 - t.done.astype(jnp.float32)

        delta = (
            t.reward
            + GAMMA * bootstrap_mask * t.next_value
            - t.value
        )

        gae = (
            delta
            + GAMMA * GAE_LAMBDA * recursion_mask * gae
        )
        return gae, gae

    _, advantages = jax.lax.scan(
        reverse_step,
        jnp.zeros((NUM_ENVS,), dtype=jnp.float32),
        traj,
        reverse=True,
    )

    returns = advantages + traj.value

    advantages = (
        advantages - advantages.mean()
    ) / (advantages.std() + 1e-8)

    return advantages, returns

def flatten_rollout(traj, advantages, returns):
    def flat(x):
        return x.reshape((BATCH_SIZE,) + x.shape[2:])

    return {
        "obs": flat(traj.obs),
        "raw_action": flat(traj.raw_action),
        "old_log_prob": flat(traj.log_prob),
        "old_value": flat(traj.value),
        "advantage": flat(advantages),
        "return": flat(returns),
    }


## SECTION 28 — PPO loss/update

The existing clipped surrogate, clipped value loss, entropy bonus, approximate KL, clip fraction, gradient clipping, shuffled minibatches, and compiled epochs are preserved. One correction is intentional: entropy now calls the existing Gaussian entropy helper instead of logging mean log-probability.


In [ ]:
def _ppo_minibatch_update(state, batch):
    def loss_fn(params):
        mean, log_std, values = network.apply(params, batch["obs"])

        actions = jnp.tanh(batch["raw_action"])
        new_log_prob = squashed_log_prob(
            batch["raw_action"], actions, mean, log_std
        )

        log_ratio = new_log_prob - batch["old_log_prob"]
        ratio = jnp.exp(log_ratio)

        advantages = batch["advantage"]
        unclipped = ratio * advantages
        clipped = (
            jnp.clip(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)
            * advantages
        )
        policy_loss = -jnp.mean(jnp.minimum(unclipped, clipped))

        value_pred_clipped = batch["old_value"] + jnp.clip(
            values - batch["old_value"],
            -CLIP_EPS,
            CLIP_EPS,
        )

        value_loss_unclipped = (values - batch["return"]) ** 2
        value_loss_clipped = (value_pred_clipped - batch["return"]) ** 2
        value_loss = 0.5 * jnp.mean(
            jnp.maximum(value_loss_unclipped, value_loss_clipped)
        )

        # Base Gaussian entropy. This is used as an exploration regularizer.
        entropy = gaussian_entropy(log_std)

        total_loss = (
            policy_loss
            + VALUE_COEF * value_loss
            - ENTROPY_COEF * entropy
        )

        approx_kl = jnp.mean(
            (jnp.exp(log_ratio) - 1.0) - log_ratio
        )
        clip_fraction = jnp.mean(
            (jnp.abs(ratio - 1.0) > CLIP_EPS).astype(jnp.float32)
        )

        aux = {
            "loss": total_loss,
            "policy_loss": policy_loss,
            "value_loss": value_loss,
            "entropy": entropy,
            "approx_kl": approx_kl,
            "clip_fraction": clip_fraction,
        }
        return total_loss, aux

    (_, aux), grads = jax.value_and_grad(
        loss_fn, has_aux=True
    )(state.params)

    state = state.apply_gradients(grads=grads)
    return state, aux

def _ppo_update_epochs(state, batch, rng):
    def epoch_step(carry, _):
        state, rng = carry
        rng, permutation_key = jax.random.split(rng)

        permutation = jax.random.permutation(
            permutation_key,
            BATCH_SIZE,
        )
        minibatch_indices = permutation.reshape(
            NUM_MINIBATCHES,
            MINIBATCH_SIZE,
        )

        def minibatch_step(state, indices):
            minibatch = jax.tree.map(
                lambda x: x[indices],
                batch,
            )
            state, metrics = _ppo_minibatch_update(
                state,
                minibatch,
            )
            return state, metrics

        state, metrics = jax.lax.scan(
            minibatch_step,
            state,
            minibatch_indices,
        )

        return (state, rng), metrics

    (state, rng), all_metrics = jax.lax.scan(
        epoch_step,
        (state, rng),
        xs=None,
        length=PPO_EPOCHS,
    )

    mean_metrics = jax.tree.map(
        jnp.mean,
        all_metrics,
    )
    return state, rng, mean_metrics

ppo_update_epochs = jax.jit(_ppo_update_epochs)


## SECTION 29 — PPO training loop

Fine-tuning starts from BC parameters with the fresh PPO optimizer. The original Python-controlled update loop and compiled hot path are retained. Checkpoints are independent of the BC checkpoint; latest and best paths are under `outputs/checkpoints`.


In [ ]:
# Fresh environment state for the BC→PPO run.
rng, reset_key = jax.random.split(rng)
env_keys = jax.random.split(reset_key, NUM_ENVS)
env_state, obs = reset_batch(env_keys)


In [ ]:
# Fresh environment state for training. The policy/optimizer may already be restored.
rng, env_key = jax.random.split(rng)
env_state, obs = reset_batch(jax.random.split(env_key, NUM_ENVS))

session_start = time.perf_counter()
session_start_steps = total_steps
if METRICS_PATH.exists():
    _historical_rewards = pd.read_csv(METRICS_PATH).get('reward', pd.Series(dtype=float))
    BEST_PPO_RETURN = float(_historical_rewards.max()) if _historical_rewards.notna().any() else -np.inf
else:
    BEST_PPO_RETURN = -np.inf

print(
    f"Training from update={update_index}, steps={total_steps:,} "
    f"toward {TOTAL_STEPS:,}"
)

try:
    while RUN_PPO_TRAINING and total_steps < TOTAL_STEPS:
        update_index += 1

        # 1) Fully compiled GPU rollout.
        env_state, obs, rng, traj = collect_rollout(
            state, env_state, obs, rng
        )


        # 2) GPU GAE and flattening.
        advantages, returns = compute_gae(traj)
        batch = flatten_rollout(traj, advantages, returns)

        # 3) All PPO epochs + minibatches are also executed through lax.scan
        #    inside one JIT-compiled update function.
        state, rng, metric_mean = ppo_update_epochs(
            state,
            batch,
            rng,
        )

        # Synchronize once per complete PPO update.
        jax.block_until_ready(state.params)

        total_steps += BATCH_SIZE

        # Only completed-episode summaries cross back to the host.
        completed_returns = np.asarray(
            jax.device_get(traj.completed_return)
        ).reshape(-1)
        completed_lengths = np.asarray(
            jax.device_get(traj.completed_length)
        ).reshape(-1)
        completed_captures = np.asarray(
            jax.device_get(traj.completed_capture)
        ).reshape(-1)

        valid = np.isfinite(completed_returns)
        episodes_finished = int(valid.sum())

        if episodes_finished:
            mean_ep_return = float(completed_returns[valid].mean())
            mean_ep_length = float(completed_lengths[valid].mean())
            capture_rate = float(completed_captures[valid].mean())
        else:
            mean_ep_return = float("nan")
            mean_ep_length = float("nan")
            capture_rate = float("nan")

        metric_mean = jax.device_get(metric_mean)

        elapsed = time.perf_counter() - session_start
        sps = (
            (total_steps - session_start_steps) / elapsed
            if elapsed > 0 else 0.0
        )

        row = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "update": update_index,
            "total_steps": total_steps,
            "reward": mean_ep_return,
            "policy_loss": float(metric_mean["policy_loss"]),
            "value_loss": float(metric_mean["value_loss"]),
            "entropy": float(metric_mean["entropy"]),
            "approx_kl": float(metric_mean["approx_kl"]),
            "clip_fraction": float(metric_mean["clip_fraction"]),
            "episode_length": mean_ep_length,
            "steps_per_second": sps,
            "both_upright": capture_rate,
            "episodes_finished": episodes_finished,
        }
        append_metric(row)
        if episodes_finished and mean_ep_return > BEST_PPO_RETURN:
            BEST_PPO_RETURN = mean_ep_return
            best_source = save_checkpoint(state, rng, update_index, total_steps)
            atomic_copy(best_source, PPO_BEST_PATH)

        if update_index % LOG_EVERY_UPDATES == 0:
            print(
                f"u={update_index:05d} "
                f"steps={total_steps:>10,} "
                f"epR={mean_ep_return:>9.2f} "
                f"len={mean_ep_length:>7.1f} "
                f"capture={capture_rate:>6.1%} "
                f"KL={float(metric_mean['approx_kl']):.4f} "
                f"SPS={sps:,.0f}"
            )

        if update_index % CHECKPOINT_EVERY_UPDATES == 0:
            path_ckpt = save_checkpoint(
                state, rng, update_index, total_steps
            )
            print("checkpoint:", path_ckpt)

except KeyboardInterrupt:
    path_ckpt = save_checkpoint(
        state, rng, update_index, total_steps
    )
    print("\nInterrupted safely.")
    print("checkpoint:", path_ckpt)
    if not PPO_BEST_PATH.exists():
        atomic_copy(path_ckpt, PPO_BEST_PATH)
    raise


# Stable artifact aliases requested by the V2 layout.
if CHECKPOINT_DIR.exists():
    final_checkpoint = save_checkpoint(state, rng, update_index, total_steps)
    if not PPO_BEST_PATH.exists():
        atomic_copy(final_checkpoint, PPO_BEST_PATH)


# PART VI — FINAL COMPARISON

## SECTION 30 — Expert vs BC vs PPO

Classical Expert, Behavior Cloning, and BC → PPO use identical seeds, initial-state distribution, success criterion, return, capture-time, cart, and control-effort definitions.


In [ ]:
ppo_episode_metrics, PPO_METRICS = evaluate_neural_policy(state.params, COMPARISON_SEEDS, "ppo")
comparison_rows = []
metric_labels = {
    "success": "Success rate",
    "return": "Mean return",
    "capture_time": "Capture time",
    "cart_rms": "Cart RMS",
    "max_cart_displacement": "Max cart travel",
    "control_effort": "Control effort",
}
for key, label in metric_labels.items():
    comparison_rows.append({"Metric": label, "Expert": EXPERT_METRICS.get(key), "BC": BC_METRICS.get(key), "PPO": PPO_METRICS.get(key)})
FINAL_COMPARISON = pd.DataFrame(comparison_rows).set_index("Metric")
FINAL_COMPARISON.to_csv(METRIC_DIR / "final_comparison.csv")
display(FINAL_COMPARISON)


## SECTION 31 — Final PPO video

Output: `outputs/videos/ppo_finetuned.mp4`, using the same showcase initial condition.


In [ ]:
ppo_video_path = VIDEO_DIR / "ppo_finetuned.mp4"
def ppo_video_policy(data, state_vector):
    obs = observation_from_data(data)[None]
    action = float(greedy_action(state.params, obs)[0, 0])
    return action * ACTION_LIMIT, "BC→PPO"
render_cpu_policy(ppo_video_path, showcase_initial_state, ppo_video_policy)
display(Video(str(ppo_video_path), embed=True))


## SECTION 32 — Comparison figures

Paper-oriented figures include BC loss, PPO return/success diagnostics, fixed expert and BC baselines, and final controller comparison.


In [ ]:
if BC_HISTORY:
    bc_frame = pd.DataFrame(BC_HISTORY)
    ax = bc_frame.plot(x="update", y=["train_loss", "validation_loss"], title="Behavior cloning loss")
    ax.figure.tight_layout(); ax.figure.savefig(PLOT_DIR / "bc_loss.png", dpi=220)

if METRICS_PATH.exists():
    ppo_frame = pd.read_csv(METRICS_PATH).drop_duplicates("update", keep="last")
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(ppo_frame.total_steps, ppo_frame.reward, label="BC→PPO")
    axes[0].axhline(EXPERT_METRICS.get("return", np.nan), ls="--", label="Expert")
    axes[0].axhline(BC_METRICS.get("return", np.nan), ls=":", label="BC")
    axes[0].set_ylabel("return"); axes[0].legend()
    axes[1].plot(ppo_frame.total_steps, ppo_frame.both_upright, label="PPO success")
    axes[1].axhline(EXPERT_METRICS.get("success", np.nan), ls="--", label="Expert")
    axes[1].axhline(BC_METRICS.get("success", np.nan), ls=":", label="BC")
    axes[1].set_ylabel("success rate"); axes[1].set_xlabel("environment steps"); axes[1].legend()
    fig.tight_layout(); fig.savefig(PLOT_DIR / "ppo_learning_curves.png", dpi=220); fig.savefig(DASHBOARD_PATH, dpi=220)

ax = FINAL_COMPARISON.T.plot.bar(subplots=True, figsize=(12, 14), legend=False, title="Final controller comparison")
plt.tight_layout(); plt.savefig(PLOT_DIR / "final_controller_comparison.png", dpi=220)
plt.show()


## SECTION 33 — Saved artifacts

```text
outputs/
    checkpoints/
        bc_pretrained.pkl
        ppo_latest.pkl
        ppo_best.pkl
    videos/
        expert_swingup_stabilization.mp4
        bc_policy.mp4
        ppo_finetuned.mp4
    plots/
        ...
    metrics/
        expert_metrics.*
        bc_metrics.*
        ppo_metrics.*
```

The manifest below records files generated by the current run.


In [ ]:
artifact_manifest = sorted(str(path.relative_to(OUTPUT_DIR)) for path in OUTPUT_DIR.rglob("*") if path.is_file())
artifact_manifest.append("artifact_manifest.json")
artifact_manifest = sorted(set(artifact_manifest))
(OUTPUT_DIR / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2))
print("\n".join(artifact_manifest))
